# Comparative Empirical Evaluation of ML Algorithms for Software Defect Prediction (NASA KC1)

Implementation of the experiment described in the paper:

- **Dataset:** NASA KC1 (raw: 2,109 modules, 21 static metrics + `defects`)
- **Preprocessing:** null audit, exact duplicate removal (→ 1,212 modules), `StandardScaler` inside a `Pipeline` (no data leakage)
- **Models:** LR, DT, RF, SVM, KNN, GB
- **Validation:** `RepeatedStratifiedKFold` 10×10 (100 folds), same folds for all 6 models, `random_state=42`
- **Metrics:** ROC-AUC, PR-AUC (threshold-independent); F1, Recall, Precision (threshold 0.5). Accuracy excluded.
- **Statistical analysis (RQ1/RQ2):** mean, SD, IQR, fold-by-fold ranking frequency, and Nadeau–Bengio corrected t-test (α = 0.05)

## 1. Imports and configuration

In [ ]:
import os
import sys
import time
import warnings

import numpy as np
import pandas as pd
from scipy import stats

IN_NOTEBOOK = "ipykernel" in sys.modules

import matplotlib
if not IN_NOTEBOOK:
    matplotlib.use("Agg")  # windowless backend when run as a script
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import RepeatedStratifiedKFold, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

warnings.filterwarnings("ignore")

# ------------------------- Configuration -------------------------
SEED = 42
N_SPLITS = 10
N_REPEATS = 10
ALPHA = 0.05
TARGET = "defects"

# Path to raw CSV. In script: python kc1_defect_prediction.py [csv_path] [output_dir]
if IN_NOTEBOOK:
    DATA_PATH = "kc1.csv"
    OUT_DIR = "results"
else:
    DATA_PATH = sys.argv[1] if len(sys.argv) > 1 else "kc1.csv"
    OUT_DIR = sys.argv[2] if len(sys.argv) > 2 else "results"

os.makedirs(OUT_DIR, exist_ok=True)
np.random.seed(SEED)

def finish_fig(filename):
    """Saves the figure; in notebook also displays it."""
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, filename), dpi=200, bbox_inches="tight")
    if IN_NOTEBOOK:
        plt.show()
    else:
        plt.close()

## 2. Data loading and auditing

Starts from the **raw** file (`kc1.csv`) so that cleaning is part of the experiment pipeline.

In [ ]:
df_raw = pd.read_csv(DATA_PATH)
df_raw[TARGET] = df_raw[TARGET].astype(int)   # True/False -> 1/0

n_feat = df_raw.shape[1] - 1
print(f"Raw dataset: {df_raw.shape[0]} modules, {n_feat} features + label '{TARGET}'")
print(f"Null values (features + label): {int(df_raw.isnull().sum().sum())}")
print(f"Exact duplicate rows:            {int(df_raw.duplicated().sum())}")

vc = df_raw[TARGET].value_counts().sort_index()
print(f"Raw distribution: non-defective = {vc[0]} ({vc[0]/len(df_raw):.2%}), "
      f"defective = {vc[1]} ({vc[1]/len(df_raw):.2%})")

## 3. Preprocessing: duplicate removal

Exact duplicates are removed keeping the first occurrence, to avoid train/test contamination
(the same record in training and validation would inflate performance).

In [ ]:
assert df_raw.isnull().sum().sum() == 0, "Expected 0 null values"

df = df_raw.drop_duplicates(keep="first").reset_index(drop=True)

X = df.drop(columns=[TARGET])
y = df[TARGET].values

vc = df[TARGET].value_counts().sort_index()
print(f"Dropped rows: {len(df_raw) - len(df)}")
print(f"Cleaned dataset:   {len(df)} modules "
      f"({vc[0]} non-defective, {vc[1]} defective; {vc[1]/len(df):.1%} defective)")

# Verification against figures reported in the paper
assert len(df) == 1212 and vc[0] == 897 and vc[1] == 315, "Does not match the paper"
print("OK: matches the paper (1,212 modules; 897 / 315; 26.0%).")

## 4. Models and pipelines

All models use the same `Pipeline` (`StandardScaler` + classifier). The scaler is fitted
only on the training fold of each split, avoiding data leakage. Tree-based models are
scale-invariant, but the same structure is kept for uniformity.

In [ ]:
models = {
    "LR":  LogisticRegression(solver="lbfgs", penalty="l2", C=1.0,
                              max_iter=1000, random_state=SEED),
    "DT":  DecisionTreeClassifier(criterion="gini", random_state=SEED),
    "RF":  RandomForestClassifier(n_estimators=200, max_features="sqrt",
                                  random_state=SEED, n_jobs=-1),
    "SVM": SVC(kernel="rbf", C=1.0, gamma="scale",
               probability=True, random_state=SEED),
    "KNN": KNeighborsClassifier(n_neighbors=5, metric="euclidean"),
    "GB":  GradientBoostingClassifier(n_estimators=100, learning_rate=0.1,
                                      random_state=SEED),
}

pipelines = {
    name: Pipeline([("scaler", StandardScaler()), ("clf", clf)])
    for name, clf in models.items()
}
print("Models:", list(pipelines))

## 5. Validation: 10×10-fold stratified

The 100 splits are generated **only once** and reused for the six classifiers, producing
fold-by-fold paired observations.

In [ ]:
rskf = RepeatedStratifiedKFold(n_splits=N_SPLITS, n_repeats=N_REPEATS, random_state=SEED)
splits = list(rskf.split(X, y))          # fixed list -> same folds for all models
n_folds = len(splits)

# Proportion of defective samples per fold (verifies stratification)
val_rates = np.array([y[te].mean() for _, te in splits])
print(f"Total folds: {n_folds}")
print(f"% defective in validation: min={val_rates.min():.3f}  max={val_rates.max():.3f}  "
      f"(global={y.mean():.3f})")

scoring = {
    "roc_auc":   "roc_auc",
    "pr_auc":    "average_precision",
    "f1":        "f1",
    "recall":    "recall",
    "precision": "precision",
}

## 6. Training and evaluation

ROC-AUC and PR-AUC are calculated using probabilities (`predict_proba`); F1, Recall, and Precision with the
standard 0.5 decision threshold.

In [ ]:
fold_scores = {}   # name -> DataFrame (100 rows x 5 metrics)
fit_time = {}

for name, pipe in pipelines.items():
    t0 = time.time()
    res = cross_validate(pipe, X, y, cv=splits, scoring=scoring,
                         n_jobs=1, error_score="raise")
    fold_scores[name] = pd.DataFrame({m: res[f"test_{m}"] for m in scoring})
    fit_time[name] = time.time() - t0
    print(f"{name:>3}: ROC-AUC = {fold_scores[name]['roc_auc'].mean():.4f}   "
          f"({fit_time[name]:.1f} s)")

# Long table with all results per fold
long = pd.concat(
    [d.assign(model=n, fold=np.arange(n_folds),
              repeat=np.arange(n_folds) // N_SPLITS) for n, d in fold_scores.items()],
    ignore_index=True,
)
long.to_csv(os.path.join(OUT_DIR, "scores_per_fold.csv"), index=False)

## 7. RQ1: performance comparison

Models are sorted by mean ROC-AUC across the 100 folds.

In [ ]:
rows = []
for name, d in fold_scores.items():
    rows.append({
        "Model": name,
        "ROC-AUC": d["roc_auc"].mean(),
        "ROC-AUC SD": d["roc_auc"].std(ddof=1),
        "PR-AUC": d["pr_auc"].mean(),
        "F1": d["f1"].mean(),
        "Recall": d["recall"].mean(),
        "Precision": d["precision"].mean(),
    })
summary = pd.DataFrame(rows).sort_values("ROC-AUC", ascending=False).reset_index(drop=True)
summary.insert(0, "Rank", np.arange(1, len(summary) + 1))
summary.to_csv(os.path.join(OUT_DIR, "table_rq1_summary.csv"), index=False)

print(summary.round(4).to_string(index=False))
summary.round(4)

## 8. RQ2: consistency across folds

- **SD** and **IQR** of ROC-AUC per model.
- **Ranking frequency:** in each fold, the 6 models are ordered by ROC-AUC (1 = best; ties with average rank) and how many times each model occupies each position is counted.

In [ ]:
auc = pd.DataFrame({n: d["roc_auc"].values for n, d in fold_scores.items()})  # 100 x 6

# --- Dispersion ---
disp = pd.DataFrame({
    "Mean": auc.mean(),
    "SD": auc.std(ddof=1),
    "IQR": auc.quantile(0.75) - auc.quantile(0.25),
    "Min": auc.min(),
    "Max": auc.max(),
}).sort_values("Mean", ascending=False)
disp.to_csv(os.path.join(OUT_DIR, "table_rq2_dispersion.csv"))
print("ROC-AUC Dispersion (100 folds):")
print(disp.round(4).to_string())

# --- Ranking frequency per fold ---
ranks = auc.rank(axis=1, ascending=False, method="average")   # 1 = best
rank_freq = pd.DataFrame(
    {r: (ranks.round().astype(int) == r).mean() for r in range(1, auc.shape[1] + 1)}
)
rank_freq.columns = [f"Rank {r}" for r in rank_freq.columns]
rank_freq["Mean Rank"] = ranks.mean()
rank_freq["% Top-1"] = (ranks == 1).mean() * 100
rank_freq = rank_freq.loc[disp.index]
rank_freq.to_csv(os.path.join(OUT_DIR, "table_rq2_ranking_frequency.csv"))
print("\nFrequency (proportion of folds) with which each model occupies each position:")
print(rank_freq.round(3).to_string())

## 9. Nadeau–Bengio corrected t-test (paired comparisons)

Repeated cross-validation folds share training data, thus violating the independence
of the classical t-test. Nadeau & Bengio correct the variance:

$$ t = \frac{\bar d}{\sqrt{\left(\frac{1}{J} + \frac{n_{test}}{n_{train}}\right)\hat\sigma_d^2}}, \qquad df = J - 1 $$

with $J = 100$ folds, $d_j$ the paired difference in ROC-AUC in fold $j$ and $\hat\sigma_d^2$ its sample variance.
Holm-adjusted p-values (15 comparisons) are also reported as supplementary information.

In [ ]:
n_train = np.mean([len(tr) for tr, _ in splits])
n_test = np.mean([len(te) for _, te in splits])
ratio = n_test / n_train
J = n_folds

def nadeau_bengio(a, b):
    d = np.asarray(a) - np.asarray(b)
    var = d.var(ddof=1)
    if var == 0:
        return 0.0, 1.0, d.mean()
    t = d.mean() / np.sqrt((1.0 / J + ratio) * var)
    p = 2 * stats.t.sf(abs(t), df=J - 1)
    return t, p, d.mean()

names = list(auc.columns)
pairs = []
for i in range(len(names)):
    for j in range(i + 1, len(names)):
        t, p, md_ = nadeau_bengio(auc[names[i]], auc[names[j]])
        pairs.append({"Model A": names[i], "Model B": names[j],
                      "Mean Diff. (A-B)": md_, "t": t, "p-value": p})

pw = pd.DataFrame(pairs)

# Holm correction (supplementary)
order = np.argsort(pw["p-value"].values)
m = len(pw)
holm = np.empty(m)
running = 0.0
for k, idx in enumerate(order):
    running = max(running, (m - k) * pw["p-value"].values[idx])
    holm[idx] = min(1.0, running)
pw["p Holm"] = holm
pw["Signif. (α=0.05, uncorr.)"] = np.where(pw["p-value"] < ALPHA, "Yes", "No")
pw["Signif. (Holm)"] = np.where(pw["p Holm"] < ALPHA, "Yes", "No")
pw = pw.sort_values("p-value").reset_index(drop=True)
pw.to_csv(os.path.join(OUT_DIR, "table_nadeau_bengio.csv"), index=False)

print(f"Mean n_train = {n_train:.1f}, Mean n_test = {n_test:.1f}, n_test/n_train = {ratio:.4f}, J = {J}")
print(pw.round(4).to_string(index=False))
n_sig = int((pw["p-value"] < ALPHA).sum())
n_sig_h = int((pw["p Holm"] < ALPHA).sum())
print(f"\nSignificant pairs: {n_sig}/{m} (uncorrected), {n_sig_h}/{m} (Holm)")
pw.round(4)

## 10. Plots

In [ ]:
order_names = list(disp.index)

# (a) ROC-AUC distribution per model
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.boxplot([auc[n] for n in order_names], showmeans=True)
ax.set_xticks(range(1, len(order_names) + 1))
ax.set_xticklabels(order_names)
ax.set_ylabel("ROC-AUC (100 folds)")
ax.set_title("ROC-AUC Distribution per Model — NASA KC1 (10×10 CV)")
ax.grid(axis="y", alpha=0.3)
finish_fig("fig_boxplot_roc_auc.png")

# (b) Ranking frequency per model
cols = [c for c in rank_freq.columns if c.startswith("Rank ") and c != "Mean Rank"]
fig, ax = plt.subplots(figsize=(8, 4.5))
rank_freq.loc[order_names, cols].plot(kind="bar", stacked=True, ax=ax, colormap="viridis_r")
ax.set_ylabel("Proportion of folds")
ax.set_xlabel("")
ax.set_title("Ranking Frequency per Fold (ROC-AUC)")
ax.legend(title="Position", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.setp(ax.get_xticklabels(), rotation=0)
finish_fig("fig_ranking_frequency.png")

# (c) Nadeau–Bengio p-value heatmap
P = pd.DataFrame(np.nan, index=order_names, columns=order_names)
for _, r in pw.iterrows():
    P.loc[r["Model A"], r["Model B"]] = r["p-value"]
    P.loc[r["Model B"], r["Model A"]] = r["p-value"]
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(P.values.astype(float), cmap="RdYlGn", vmin=0, vmax=0.2)
ax.set_xticks(range(len(order_names))); ax.set_xticklabels(order_names)
ax.set_yticks(range(len(order_names))); ax.set_yticklabels(order_names)
for i in range(len(order_names)):
    for j in range(len(order_names)):
        v = P.values[i, j]
        if not np.isnan(v):
            ax.text(j, i, f"{v:.3f}", ha="center", va="center", fontsize=8)
ax.set_title("Nadeau–Bengio p-values (ROC-AUC)\ngreen = more evidence of difference")
fig.colorbar(im, ax=ax, label="p-value (scale truncada a 0.2)")
finish_fig("fig_heatmap_pvalues.png")

## 11. Final summary

In [ ]:
best = summary.iloc[0]
print("=" * 70)
print("SUMMARY")
print("=" * 70)
print(f"Cleaned dataset: {len(df)} modules ({y.mean():.1%} defective) | "
      f"{J} folds ({N_SPLITS}×{N_REPEATS}) | seed={SEED}")
print(f"\nRQ1 — Best mean ROC-AUC: {best['Model']} = {best['ROC-AUC']:.4f} "
      f"(SD {best['ROC-AUC SD']:.4f})")
print("Ranking by mean ROC-AUC: " + " > ".join(summary["Model"]))
print(f"\nRQ2 — Pairs with significant difference (Nadeau–Bengio, α={ALPHA}): "
      f"{n_sig}/{m} uncorrected; {n_sig_h}/{m} with Holm")
print(f"Model with highest % of folds in 1st position: "
      f"{rank_freq['% Top-1'].idxmax()} ({rank_freq['% Top-1'].max():.0f}% of folds)")
print(f"\nFiles saved in: {os.path.abspath(OUT_DIR)}")